# ЛР 01.1 — Filter Methods для оценки значимости признаков (TODO)

Ориентир по времени: 90 минут.

## Цель
Сформировать базовый `feature_ranking` двумя группами filter-методов и получить первичный shortlist признаков для двух прикладных датасетов.

## Что важно
- отбор выполняется **только по train-части**;
- один и тот же формат результатов для обоих датасетов;
- фиксируем не только код, но и интерпретацию.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    evaluate_binary_model,
    append_ranking_rows,
    build_shortlist,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Шаг 1. Загрузка данных
Проверьте размеры, долю положительного класса и общий вид колонок.

In [ ]:
dataset_frames = {}
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    dataset_frames[dataset_name] = df
    print(f"{dataset_name}: shape={df.shape}, target_mean={df['target'].mean():.3f}")

display(dataset_frames['medical'].head(3))
display(dataset_frames['finance'].head(3))

## Шаг 2. Единый baseline на полном наборе признаков
Используем `LogisticRegression` в pipeline с общим препроцессингом для числовых и категориальных признаков.

In [ ]:
baseline_rows = []
prepared = {}

for dataset_name, df in dataset_frames.items():
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    baseline_model = Pipeline(
        steps=[
            ('preprocessor', build_preprocessor(X_train)),
            ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)),
        ]
    )
    baseline_metrics = evaluate_binary_model(
        baseline_model,
        X_train,
        y_train,
        X_test,
        y_test,
    )
    baseline_rows.append({'dataset': dataset_name, **baseline_metrics})

    preprocessor_fs = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor_fs, X_train, X_test)
    prepared[dataset_name] = {
        'X_train_t': X_train_t,
        'X_test_t': X_test_t,
        'y_train': y_train,
        'feature_names': feature_names,
    }

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

## Шаг 3. Filter-методы
Применяем:
- `VarianceThreshold`
- абсолютную корреляцию с таргетом
- `mutual_info_classif`
- `f_classif`

TODO: измените `threshold`, `top_n` и сравните, как меняется shortlist.

In [ ]:
ranking_rows = []

for dataset_name, bundle in prepared.items():
    X_train_t = to_dense(bundle['X_train_t'])
    y_train = bundle['y_train'].to_numpy()
    feature_names = bundle['feature_names']

    # 1) Variance
    threshold = 0.01
    var_selector = VarianceThreshold(threshold=threshold)
    var_selector.fit(X_train_t)
    variance_scores = var_selector.variances_
    append_ranking_rows(ranking_rows, dataset_name, 'variance', feature_names, variance_scores)

    # 2) Abs correlation with target
    corr_scores = []
    for idx in range(X_train_t.shape[1]):
        col = X_train_t[:, idx]
        if np.std(col) < 1e-12:
            corr_scores.append(0.0)
            continue
        corr_value = np.corrcoef(col, y_train)[0, 1]
        corr_scores.append(abs(corr_value) if np.isfinite(corr_value) else 0.0)
    corr_scores = np.array(corr_scores)
    append_ranking_rows(ranking_rows, dataset_name, 'abs_correlation', feature_names, corr_scores)

    # 3) Mutual information
    mi_scores = mutual_info_classif(X_train_t, y_train, random_state=SEED)
    append_ranking_rows(ranking_rows, dataset_name, 'mutual_info', feature_names, mi_scores)

    # 4) ANOVA F-test
    f_scores, _ = f_classif(X_train_t, y_train)
    f_scores = np.nan_to_num(f_scores, nan=0.0, posinf=0.0, neginf=0.0)
    append_ranking_rows(ranking_rows, dataset_name, 'f_classif', feature_names, f_scores)

feature_ranking = (
    pd.DataFrame(ranking_rows)
    .sort_values(['dataset', 'method', 'rank'])
    .reset_index(drop=True)
)
feature_ranking.head(20)

## Шаг 4. Формирование shortlist
Берем средний ранг по выбранным методам и оставляем топ признаков.

In [ ]:
filter_methods = ['variance', 'abs_correlation', 'mutual_info', 'f_classif']
shortlists = {}

for dataset_name in prepared:
    top_n = 12
    shortlists[dataset_name] = build_shortlist(
        feature_ranking=feature_ranking,
        dataset_name=dataset_name,
        methods=filter_methods,
        top_n=top_n,
    )

shortlists

## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о методах/подходах, которые вы изучили именно на этом этапе.
- Укажите, почему эти идеи важны для текущего шага ноутбука.

### Сравнение с альтернативами
- TODO (обязательно): минимум одно сравнение с альтернативным методом/подходом.
- Формат: "когда лучше метод A, когда лучше метод B, и почему".

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).
- Если объяснение не помещается в ноутбук: вынесите разбор в `study-notes/*.md`
  и оставьте здесь явную ссылку на файл.

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): по ходу этого шага добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- Для каждого термина укажите: простое объяснение, где встретился термин в ноутбуке, источник.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.


In [ ]:
feature_ranking_path = OUTPUT_DIR / 'feature_ranking_filter_methods.csv'
shortlist_path = OUTPUT_DIR / 'shortlist_filter.json'

feature_ranking.to_csv(feature_ranking_path, index=False)
with open(shortlist_path, 'w', encoding='utf-8') as f:
    json.dump(shortlists, f, ensure_ascii=False, indent=2)

print(f'Saved: {feature_ranking_path}')
print(f'Saved: {shortlist_path}')

## Контрольные точки
1. В `feature_ranking` есть колонки `dataset, method, feature, score, rank`.
2. Для каждого датасета присутствуют 4 метода.
3. Shortlist для каждого датасета не пустой.

In [ ]:
required_columns = {'dataset', 'method', 'feature', 'score', 'rank'}
assert required_columns.issubset(feature_ranking.columns)

for dataset_name in DATASETS:
    subset = feature_ranking[feature_ranking['dataset'] == dataset_name]
    assert subset['method'].nunique() == 4
    assert len(shortlists[dataset_name]) > 0

assert feature_ranking['score'].notna().all()
assert feature_ranking['rank'].notna().all()

print('Проверки пройдены успешно.')

In [ ]:
mean_rank = (
    feature_ranking
    .groupby(['dataset', 'feature'], as_index=False)['rank']
    .mean()
    .sort_values(['dataset', 'rank'])
)

plt.figure(figsize=(12, 5))
sns.barplot(
    data=mean_rank.groupby('dataset').head(8),
    x='rank',
    y='feature',
    hue='dataset',
    orient='h',
)
plt.title('Топ-8 признаков по среднему рангу (меньше = лучше)')
plt.xlabel('Средний ранг')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

## Типичные ошибки
- Отбор признаков до train/test split (утечка таргета).
- Сравнение методов по несопоставимым шкалам score.
- Использование только одного метода без проверки устойчивости топа.

## Финальные выводы (заполните)
TODO:
1. Какие 5-7 признаков чаще всего оказываются наверху рангов для `medical`?
2. Какие 5-7 признаков наиболее важны для `finance`?
3. Чем отличаются выводы разных filter-методов и почему это нормально?

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
Каждый блок содержит intentional-stop: ноутбук останавливается, пока вы не заполните шаблон своим кодом.

### Задание 1. Устойчивость shortlist на сетке параметров

**Контракт задания**

Входные переменные:
- `feature_ranking`, `prepared`, `build_shortlist`, `OUTPUT_DIR`.

Ожидаемый выход:
- `filter_stability_grid` с колонками:
  `dataset`, `variance_threshold`, `top_n`, `shortlist_json`, `overlap_with_baseline`.

Критерий завершения:
- таблица заполнена для всех конфигураций grid;
- колонки проходят `assert required_columns...`.

In [ ]:
required_columns_task1 = [
    'dataset',
    'variance_threshold',
    'top_n',
    'shortlist_json',
    'overlap_with_baseline',
]
filter_stability_grid = pd.DataFrame(columns=required_columns_task1)
assert set(required_columns_task1).issubset(filter_stability_grid.columns)

variance_threshold_grid = [0.0, 0.005, 0.01, 0.02]
top_n_grid = [8, 10, 12]

# TODO(обязательно):
# 1) Для каждого dataset и каждой пары (variance_threshold, top_n)
#    вычислите shortlist через build_shortlist.
# 2) baseline берем для (variance_threshold=0.01, top_n=12).
# 3) overlap_with_baseline = |A ∩ B|.
# 4) shortlist_json сохраните как строку JSON.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 1 не завершено: заполните DataFrame filter_stability_grid по required_columns_task1.'
)

### Задание 2. Pairwise overlap и Jaccard между конфигурациями

**Контракт задания**

Входные переменные:
- `filter_stability_grid` из задания 1.

Ожидаемый выход:
- `filter_pairwise_similarity` с колонками:
  `dataset`, `config_a`, `config_b`, `overlap_count`, `jaccard`.

Критерий завершения:
- для каждого dataset рассчитаны попарные метрики между всеми конфигурациями.

In [ ]:
required_columns_task2 = [
    'dataset',
    'config_a',
    'config_b',
    'overlap_count',
    'jaccard',
]
filter_pairwise_similarity = pd.DataFrame(columns=required_columns_task2)
assert set(required_columns_task2).issubset(filter_pairwise_similarity.columns)

# TODO(обязательно):
# 1) Восстановите shortlist из shortlist_json.
# 2) Для каждой пары конфигураций в пределах одного dataset посчитайте:
#    overlap_count = |A ∩ B|,
#    jaccard = |A ∩ B| / |A ∪ B|.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 2 не завершено: заполните DataFrame filter_pairwise_similarity по required_columns_task2.'
)

### Задание 3. Экспорт и краткий вывод

**Контракт задания**

Входные переменные:
- `filter_stability_grid` из задания 1.

Ожидаемый выход:
- файл `outputs/filter_stability_grid.csv`;
- 2-3 коротких вывода для `medical` и `finance` о наиболее устойчивых конфигурациях.

In [ ]:
filter_stability_path = OUTPUT_DIR / 'filter_stability_grid.csv'
required_columns_task1 = {
    'dataset',
    'variance_threshold',
    'top_n',
    'shortlist_json',
    'overlap_with_baseline',
}

# TODO(обязательно):
# 1) Проверьте наличие required columns в filter_stability_grid.
# 2) Сохраните filter_stability_grid в CSV.
# 3) Напечатайте сводку по устойчивости отдельно для medical и finance.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

raise NotImplementedError(
    'Задание 3 не завершено: сохраните outputs/filter_stability_grid.csv и добавьте интерпретацию.'
)